In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# DEVICE
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# PRUNABLE LINEAR

class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

        # Balanced initialization (not too small, not too big)
        self.gate_scores = nn.Parameter(torch.randn_like(self.weight) - 1)

        # Moderate sharpness
        self.temperature = 0.5

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores / self.temperature)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self):
        return torch.sigmoid(self.gate_scores / self.temperature)


In [4]:
# MODEL
class PrunableMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.Sequential(
            PrunableLinear(3072, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),

            PrunableLinear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            PrunableLinear(512, 256),
            nn.ReLU(),

            PrunableLinear(256, 10)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.layers(x)

    def get_all_gates(self):
        return [m.get_gates() for m in self.modules() if isinstance(m, PrunableLinear)]


In [5]:
# LOSS

def compute_loss(model, outputs, targets, lambda_sparsity):
    ce_loss = F.cross_entropy(outputs, targets)

    sparsity_loss = 0.0
    for gates in model.get_all_gates():
        sparsity_loss += gates.mean()

    total_loss = ce_loss + lambda_sparsity * sparsity_loss
    return total_loss, ce_loss, sparsity_loss


In [6]:
# METRICS

def compute_sparsity(model, threshold=0.1):
    total = 0
    zero = 0

    for gates in model.get_all_gates():
        total += gates.numel()
        zero += (gates < threshold).sum().item()

    return 100.0 * zero / total


def accuracy(outputs, targets):
    preds = outputs.argmax(dim=1)
    return (preds == targets).float().mean().item()


In [7]:
# TRAIN

def train(model, loader, optimizer, device, lambda_sparsity):
    model.train()

    total_loss, total_acc = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss, ce_loss, sp_loss = compute_loss(model, outputs, y, lambda_sparsity)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(outputs, y)

    return total_loss / len(loader), total_acc / len(loader)

In [8]:
# EVAL

def evaluate(model, loader, device):
    model.eval()
    total_acc = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            total_acc += accuracy(outputs, y)

    return total_acc / len(loader)

In [9]:
# DATA

def get_dataloaders(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train_data = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    test_data = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size)

    return train_loader, test_loader

In [ ]:
# RUN

def run_experiment(lambda_sparsity):
    device = get_device()
    model = PrunableMLP().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_loader, test_loader = get_dataloaders()

    for epoch in range(10):
        train_loss, train_acc = train(model, train_loader, optimizer, device, lambda_sparsity)
        test_acc = evaluate(model, test_loader, device)
        sparsity = compute_sparsity(model)

        print(f"[λ={lambda_sparsity}] Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Test Acc: {test_acc:.4f} | Sparsity: {sparsity:.2f}%")
        print("-" * 50)

    return test_acc, sparsity


if __name__ == "__main__":

    lambdas = [1e-4, 5e-4, 1e-3]

    results = []
    for lam in lambdas:
        acc, sp = run_experiment(lam)
        results.append((lam, acc, sp))

    print("\nFinal Results:")
    for lam, acc, sp in results:
        print(f"Lambda: {lam} | Accuracy: {acc:.4f} | Sparsity: {sp:.2f}%")

100%|██████████| 170M/170M [00:02<00:00, 61.4MB/s]


[λ=0.0001] Epoch 1
Train Loss: 2.0295 | Train Acc: 0.2334
Test Acc: 0.3134 | Sparsity: 46.15%
--------------------------------------------------
[λ=0.0001] Epoch 2
Train Loss: 1.8148 | Train Acc: 0.3367
Test Acc: 0.3743 | Sparsity: 46.21%
--------------------------------------------------
